# 02 · Train the rule baseline and Isolation Forest

The rule is fit **first** and frozen, so no model is tuned to beat a moving baseline. Both fit on normal-train only; both take their threshold from held-out normal at the same false-positive budget. Artefacts and manifests land in `../models/` (git-lfs).

In [ ]:
from pathlib import Path

import pandas as pd

import features

META = ["window_start_ns", "window_end_ns", "label", "fault_kind", "severity", "command_id"]
PARQUET = Path("..") / "data" / f"train-{features.__version__}.parquet"

df = pd.read_parquet(PARQUET)
feature_cols = [c for c in df.columns if c not in META]
X = df[feature_cols].to_numpy(dtype=float)
kinds = df["fault_kind"].tolist()
starts = df["window_start_ns"].to_numpy()
print(f"{len(df)} windows, {len(feature_cols)} features, features v{features.__version__}")

In [ ]:
import numpy as np

# Deterministic time-based split (no RNG), identical in every notebook: the
# earliest 70% of normal windows train the detectors; the rest, plus every
# fault window, form the evaluation set. Training on earlier-normal and
# testing on later-normal is the honest ordering for a live detector.
order = list(np.argsort(starts))
normal_positions = [i for i in order if kinds[i] == "none"]
cut = int(0.7 * len(normal_positions))
train_rows = normal_positions[:cut]
holdout_rows = normal_positions[cut:]
eval_rows = holdout_rows + [i for i in order if kinds[i] != "none"]

X_train = X[train_rows]
X_holdout = X[holdout_rows]
X_eval = X[eval_rows]
kinds_eval = [kinds[i] for i in eval_rows]
print(
    f"train(normal)={len(train_rows)}  eval={len(eval_rows)} "
    f"(holdout normal={len(holdout_rows)}, faults={len(eval_rows) - len(holdout_rows)})"
)

In [ ]:
from datetime import date

from detector import (
    Manifest,
    choose_threshold,
    dataset_fingerprint,
    evaluate,
    fit_forest,
    fit_rule,
    save_detector,
)

SEED = 0
MODELS = Path("..") / "models"
fingerprint = dataset_fingerprint(PARQUET)
today = date.today().isoformat()

## Rule baseline (frozen first)

In [ ]:
rule = fit_rule(X_train, feature_cols)
rule_threshold = choose_threshold(rule.score(X_holdout))
rule_eval = evaluate(rule.score(X_eval), kinds_eval, rule_threshold)
save_detector(
    rule,
    Manifest(
        "rule-v1",
        "rule",
        features.__version__,
        rule_threshold,
        SEED,
        today,
        fingerprint,
        len(train_rows),
        rule_eval.as_metrics(),
    ),
    MODELS,
)
print("rule-v1:", rule_eval.as_metrics())

## Isolation Forest

In [ ]:
forest = fit_forest(X_train, feature_cols, seed=SEED)
forest_threshold = choose_threshold(forest.score(X_holdout))
forest_eval = evaluate(forest.score(X_eval), kinds_eval, forest_threshold)
save_detector(
    forest,
    Manifest(
        "isoforest-v1",
        "forest",
        features.__version__,
        forest_threshold,
        SEED,
        today,
        fingerprint,
        len(train_rows),
        forest_eval.as_metrics(),
    ),
    MODELS,
)
print("isoforest-v1:", forest_eval.as_metrics())